<a href="https://colab.research.google.com/github/Artur210902/itmo_ml_for_science_course/blob/HW-2/HW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("umuttuygurr/e-commerce-customer-behavior-and-sales-analysis-tr")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'e-commerce-customer-behavior-and-sales-analysis-tr' dataset.
Path to dataset files: /kaggle/input/e-commerce-customer-behavior-and-sales-analysis-tr


In [7]:
import pandas as pd
import os

# Получаем путь к файлу CSV из загруженных данных
# Предполагаем, что CSV файл находится непосредственно в загруженной папке
# Проверим содержимое папки для точного имени файла
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]

if len(csv_files) > 0:
    csv_file_path = os.path.join(path, csv_files[0])
    df = pd.read_csv(csv_file_path)

    # Выводим первые строки DataFrame и информацию о нем
    print("Первые 5 строк DataFrame:")
    display(df.head())

    print("\nИнформация о DataFrame:")
    df.info()
else:
    print("В загруженной папке не найдено ни одного CSV файла.")

Первые 5 строк DataFrame:


,Order_ID,Customer_ID,Date,Age,Gender,City,Product_Category,Unit_Price,Quantity,Discount_Amount,Total_Amount,Payment_Method,Device_Type,Session_Duration_Minutes,Pages_Viewed,Is_Returning_Customer,Delivery_Time_Days,Customer_Rating
0,ORD_001337,CUST_01337,2023-01-01,27,Female,Bursa,Toys,54.28,1,0.00,54.28,Debit Card,Mobile,4,14,True,8,5
1,ORD_004885,CUST_04885,2023-01-01,42,Male,Konya,Toys,244.90,1,0.00,244.90,Credit Card,Mobile,11,3,True,3,3
2,ORD_004507,CUST_04507,2023-01-01,43,Female,Ankara,Food,48.15,5,0.00,240.75,Credit Card,Mobile,7,8,True,5,2
3,ORD_000645,CUST_00645,2023-01-01,32,Male,Istanbul,Electronics,804.06,1,229.28,574.78,Credit Card,Mobile,8,10,False,1,4
4,ORD_000690,CUST_00690,2023-01-01,40,Female,Istanbul,Sports,755.61,5,0.00,3778.05,Cash on Delivery,Desktop,21,10,True,7,4



Информация о DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Order_ID                  5000 non-null   object 
 1   Customer_ID               5000 non-null   object 
 2   Date                      5000 non-null   object 
 3   Age                       5000 non-null   int64  
 4   Gender                    5000 non-null   object 
 5   City                      5000 non-null   object 
 6   Product_Category          5000 non-null   object 
 7   Unit_Price                5000 non-null   float64
 8   Quantity                  5000 non-null   int64  
 9   Discount_Amount           5000 non-null   float64
 10  Total_Amount              5000 non-null   float64
 11  Payment_Method            5000 non-null   object 
 12  Device_Type               5000 non-null   object 
 13  Session_Duration_Minutes  5000 non-nul

### Разбиение данных на тренировочную и тестовую выборки

Для оценки качества модели на unseen данных, разобьем исходный датасет на тренировочную и тестовую выборки. Это позволит избежать переобучения и получить более объективную оценку производительности модели.

In [9]:
from sklearn.model_selection import train_test_split

# Определим целевую переменную (например, 'Total_Amount' для регрессии)
# В данном случае, судя по заданию, мы можем предсказывать что-то связанное с поведением клиента или продажами.
# В качестве примера выберем 'Total_Amount' как целевую переменную для задачи регрессии.

# Для задачи регрессии:
target_variable = 'Total_Amount'
X = df.drop(target_variable, axis=1)
y = df[target_variable]



# Разбиваем данные на тренировочную и тестовую выборки
# Устанавливаем random_state для воспроизводимости
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Размер тренировочной выборки:", X_train.shape)
print("Размер тестовой выборки:", X_test.shape)

Размер тренировочной выборки: (4000, 17)
Размер тестовой выборки: (1000, 17)


### Расчет качества константного предсказания (бейзлайн)

В качестве простого бейзлайна для задачи регрессии будем использовать константное предсказание, равное среднему значению целевой переменной ('Total_Amount') на тренировочной выборке. Оценим качество этого предсказания на тестовой выборке, используя метрику Средняя абсолютная ошибка (MAE).

In [11]:
from sklearn.metrics import mean_absolute_error
import numpy as np

# Рассчитываем среднее значение целевой переменной на тренировочной выборке
mean_total_amount_train = np.mean(y_train)

# Создаем массив константных предсказаний для тестовой выборки
y_pred_baseline_constant = np.full_like(y_test, mean_total_amount_train)

# Оцениваем качество константного предсказания с помощью MAE
mae_baseline_constant = mean_absolute_error(y_test, y_pred_baseline_constant)

print(f"Среднее значение 'Total_Amount' на тренировочной выборке: {mean_total_amount_train:.2f}")
print(f"MAE для константного предсказания на тестовой выборке: {mae_baseline_constant:.2f}")

Среднее значение 'Total_Amount' на тренировочной выборке: 1023.77
MAE для константного предсказания на тестовой выборке: 937.67


### Обучение и оценка бейзлайн-модели (Линейная регрессия)

В качестве бейзлайн-модели выберем модель Линейной регрессии, так как это простое и интерпретируемое семейство моделей, подходящее для задачи регрессии. Перед обучением модели необходимо выполнить предобработку данных:
1. **Обработка категориальных признаков**: Преобразуем категориальные признаки в числовой формат с помощью One-Hot Encoding.
2. **Масштабирование числовых признаков**: Масштабируем числовые признаки для улучшения сходимости модели.

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Определим неинформативные признаки для удаления
features_to_drop = ['Order_ID', 'Customer_ID', 'Date']

# Удалим неинформативные признаки из X_train и X_test
X_train_processed = X_train.drop(features_to_drop, axis=1)
X_test_processed = X_test.drop(features_to_drop, axis=1)

# Определим числовые и категориальные признаки после удаления неинформативных
numerical_features = X_train_processed.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train_processed.select_dtypes(include=['object', 'bool']).columns.tolist()

# Создаем препроцессор для обработки числовых и категориальных признаков
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='drop' # Удаляем остальные колонки
)

# Создаем пайплайн: сначала препроцессинг, затем модель линейной регрессии
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('regressor', LinearRegression())])

# Обучаем модель на тренировочной выборке
model_pipeline.fit(X_train_processed, y_train)

# Делаем предсказания на тестовой выборке
y_pred_baseline_model = model_pipeline.predict(X_test_processed)

# Оцениваем качество модели на тестовой выборке
mae_baseline_model = mean_absolute_error(y_test, y_pred_baseline_model)
mse_baseline_model = mean_squared_error(y_test, y_pred_baseline_model)
rmse_baseline_model = np.sqrt(mse_baseline_model)
r2_baseline_model = r2_score(y_test, y_pred_baseline_model)


print(f"MAE для модели Линейной регрессии на тестовой выборке: {mae_baseline_model:.2f}")
print(f"MSE для модели Линейной регрессии на тестовой выборке: {mse_baseline_model:.2f}")
print(f"RMSE для модели Линейной регрессии на тестовой выборке: {rmse_baseline_model:.2f}")
print(f"R2 score для модели Линейной регрессии на тестовой выборке: {r2_baseline_model:.2f}")

MAE для модели Линейной регрессии на тестовой выборке: 484.02
MSE для модели Линейной регрессии на тестовой выборке: 579110.98
RMSE для модели Линейной регрессии на тестовой выборке: 760.99
R2 score для модели Линейной регрессии на тестовой выборке: 0.73


### Выводы по бейзлайн-моделям

Мы сравнили качество двух бейзлайн-моделей для предсказания `Total_Amount`:

1.  **Константное предсказание**: Эта простейшая модель предсказывает среднее значение `Total_Amount` на тренировочной выборке для всех примеров в тестовой выборке.
    *   MAE на тестовой выборке: 937.67

2.  **Линейная регрессия**: Эта модель была обучена на предобработанных данных (с удалением неинформативных признаков, масштабированием числовых и One-Hot Encoding категориальных признаков).
    * MAE для модели Линейной регрессии на тестовой выборке: 484.02
    * MSE для модели Линейной регрессии на тестовой выборке: 579110.98
    * RMSE для модели Линейной регрессии на тестовой выборке: 760.99
    * R2 score для модели Линейной регрессии на тестовой выборке: 0.73

**Сравнение и выводы:**

Как видно по метрикам, модель Линейной регрессии показывает значительно лучшее качество предсказания по сравнению с константным предсказанием. Это подтверждается также более высоким значением R2 score, которое показывает, что линейная модель объясняет 73% дисперсии целевой переменной на тестовой выборке.

Таким образом, модель Линейной регрессии успешно превзошла простой константный бейзлайн и может служить отправной точкой для дальнейшего улучшения качества предсказаний с помощью более сложных моделей.